# SHIPP Stage 13 — Marketplace Analytics

Lakebase `saved_items` + `agent_activity` CDF history → Spark current-state reconstruction → `gold_marketplace_metrics`.

**P0 evidence goal:** before approved save = X, after save = X + 1, rerun with no new write = still X + 1.

This notebook writes exactly one current metrics snapshot row (`metric_key = marketplace_current`). It is deliberately small: no dashboard styling.


In [ ]:
%run ../common/shipp_silver_lib

In [ ]:
from data_pipeline.gold.analytics import METRIC_COLUMNS, build_marketplace_metrics
from config.tables import (
    BRONZE_SAVED_ITEMS_HISTORY,
    BRONZE_AGENT_ACTIVITY_HISTORY,
    GOLD_MARKETPLACE_METRICS,
)

print('saved_items CDF =', BRONZE_SAVED_ITEMS_HISTORY)
print('agent_activity CDF =', BRONZE_AGENT_ACTIVITY_HISTORY)
print('target =', GOLD_MARKETPLACE_METRICS)


## Gate 1 — CDF sources exist and contain the expected keys


In [ ]:
saved_history = spark.table(BRONZE_SAVED_ITEMS_HISTORY)
activity_history = spark.table(BRONZE_AGENT_ACTIVITY_HISTORY)

saved_required = {'saved_item_id', '_pg_change_type', '_pg_lsn', '_sort_by'}
activity_required = {'activity_id', 'tool_name', 'action_status', '_pg_change_type', '_pg_lsn', '_sort_by'}

missing_saved = saved_required - set(saved_history.columns)
missing_activity = activity_required - set(activity_history.columns)
assert not missing_saved, f'Missing saved_items CDF columns: {sorted(missing_saved)}'
assert not missing_activity, f'Missing agent_activity CDF columns: {sorted(missing_activity)}'

print('saved_items CDF rows:', saved_history.count())
print('agent_activity CDF rows:', activity_history.count())
print('PASS — analytics CDF sources are readable.')


## Gate 2 — Build idempotent current metrics snapshot


In [ ]:
metrics_df = build_marketplace_metrics(saved_history, activity_history)
assert metrics_df.count() == 1
assert metrics_df.columns == METRIC_COLUMNS
display(metrics_df)
print('PASS — marketplace metrics snapshot built.')


## Gate 3 — Persist one Gold snapshot row with MERGE


In [ ]:
spark.sql('CREATE SCHEMA IF NOT EXISTS bootcamp_students.shipp_gold')

if not spark.catalog.tableExists(GOLD_MARKETPLACE_METRICS):
    metrics_df.limit(0).write.format('delta').mode('overwrite').saveAsTable(GOLD_MARKETPLACE_METRICS)

metrics_df.createOrReplaceTempView('_shipp_marketplace_metrics_source')
spark.sql(f'''\
MERGE INTO {GOLD_MARKETPLACE_METRICS} AS t
USING _shipp_marketplace_metrics_source AS s
ON t.metric_key = s.metric_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
''')

persisted = spark.table(GOLD_MARKETPLACE_METRICS)
assert persisted.filter("metric_key = 'marketplace_current'").count() == 1
display(persisted)
print('PASS — gold_marketplace_metrics contains exactly one current snapshot row.')


## Gate 4 — Final rubric evidence

Capture these three runs:

1. **Before save:** note `saved_item_count = X`.
2. Execute one new approved `save_item()` and wait for Lakebase CDF to publish the `saved_items` and `agent_activity` events.
3. **After save:** rerun this notebook and prove `saved_item_count = X + 1`.
4. **Idempotency:** rerun again without another save and prove `saved_item_count` is unchanged at `X + 1`.

Also capture the relevant `agent_activity` counts for SUCCESS / REJECTED / DUPLICATE / FAILED as applicable.


In [ ]:
row = spark.table(GOLD_MARKETPLACE_METRICS).filter("metric_key = 'marketplace_current'").first()
print({
    'saved_item_count': row['saved_item_count'],
    'agent_action_count': row['agent_action_count'],
    'save_item_action_count': row['save_item_action_count'],
    'agent_success_count': row['agent_success_count'],
    'agent_failure_count': row['agent_failure_count'],
    'agent_rejected_count': row['agent_rejected_count'],
    'agent_duplicate_count': row['agent_duplicate_count'],
    'source_saved_max_lsn': row['source_saved_max_lsn'],
    'source_activity_max_lsn': row['source_activity_max_lsn'],
})
